In [1]:
import pandas as pd

In [23]:
import numpy as np

_k = 3
_ret = 'bm25'
_dataset = 'dev_small'

res_name = f'{_ret}_{_dataset}'
experiment_name = f'{res_name}_integrated_{_k}'
retr_res = pd.read_csv(f'../../rag_utility/res/{res_name}.csv')
qualt5_df = pd.read_csv(f'./quality_res/{experiment_name}.csv')
qualt5_df.qid = qualt5_df.qid.astype('str')

# qualt5_df = qualt5_df[qualt5_df['rank']<_k]
qualt5_df.quality = qualt5_df.quality.apply(lambda x: np.exp(x))

# temp_df = qualt5_df.groupby(['qid']).quality.apply(lambda x: x.max())
# max_score_dict = dict(zip(temp_df.index.tolist(), list(temp_df.values)))

score_dict = dict(zip(qualt5_df.qid, qualt5_df.quality))

In [24]:
import json
import numpy as np

if('nq' in _dataset):
    _dataset_dev, _prefix, _suffix = _dataset, 'short', 'concise'
else:
    _dataset_dev, _prefix, _suffix = 'dev_small', 'random', 'prompt1'

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_1calls_0_0_bm25_dl_{_dataset_dev}_{_suffix}_eval.json')
zero_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}_eval.json')
k_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}.json')
k_gens = json.load(f)
f.close()

f = open(f'../coherence_eval/log_prob_temp_res/full_context/{_dataset_dev}_{_ret}_{_k}.json')
perpC = json.load(f)
f.close()

dev_res = retr_res[['qid', 'query']].drop_duplicates().copy()
dev_res.qid = dev_res.qid.astype('str')
dev_res = dev_res[dev_res.qid.isin(qualt5_df.qid.unique())]
dev_res['quality'] = dev_res.qid.apply(lambda x: score_dict[x])

if('nq' in _dataset):
    base_f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items() if ('0' in item[1].keys())}
else:
    base_f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in k_evals.items() if ('0' in item[1].keys())}

kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

dev_res = dev_res[dev_res.qid.astype('str').isin(f1_dict.keys())]
dev_res['f1'] = dev_res.qid.apply(lambda x: f1_dict[str(x)])
dev_res['utility'] = dev_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict[str(x)])

dev_res = dev_res.dropna(axis='index')
dev_res.head(3)

,qid,query,quality,f1,utility
0,1048585,what is paula deen s brother,0.192350,0.592672,0.295708
100,2,androgen receptor define,0.630692,0.583634,-0.045114
200,524332,treating tension headaches without medication,0.295599,0.519813,0.026864


In [25]:
from scipy import stats

stats.spearmanr(dev_res.quality, dev_res.utility)

SignificanceResult(statistic=np.float64(0.1256426124062371), pvalue=np.float64(5.934651897140583e-26))